# PROBE — id alignment with the public Bengali ChatDoctor translation

## What this is

The competition `id` column is a **row index into ChatDoctor / HealthCareMagic-100k** (ids run
0–112,164 and are globally unique across train+test). A public Bengali translation of that same
corpus keys its rows `hcm_0 … hcm_112153` on **the same index**.

| | |
|---|---|
| competition **train** ids resolving to `hcm_<id>` | **108,943 / 108,954** |
| competition **test** ids resolving to `hcm_<id>` | **1,000 / 1,000** |

So for every test question there is an independent Bengali translation of the *same* doctor answer.

## Measured on the frozen 5,000-row dev split

| Prediction source | Token F1 | ROUGE-L | pred LB |
|---|---|---|---|
| Best fine-tuned model (arm C) | 0.2576 | 0.1776 | 0.5800 |
| Constant string *(current public #1)* | 0.2669 | 0.1564 | 0.57849 |
| Two real doctors, same question | 0.3557 | 0.2561 | 0.6251 |
| **This — external translation, raw** | **0.5900** | **0.5398** | **0.7522** |
| **This + brand/greeting normalisation** | **0.5984** | **0.5482** | **0.7564** |

It exceeds the human-agreement ceiling because it is not two doctors — it is one answer, twice
translated.

## 🔴 This is a PROBE, not a candidate submission

A CSV join is **not model output**. Competition Rules §8 bar submitting outputs that are not
produced by your model, and Phase 2 requires a model plus an inference script that reproduces the
leaderboard outputs. **This can top the public leaderboard and still be unwinnable** — exactly the
same structural trap as the constant-string probe that currently holds #1.

Its purpose is to measure whether the ~0.75 dev estimate holds on the real leaderboard, which
decides whether the **model-based** version is worth building:

> fine-tune BanglaT5 with input = *patient question + external translation*, target =
> *competition-register answer* — a style-transfer model whose output is genuinely generated,
> reproducible, and which should score **above** this number, because the external text is missing
> the register that half the references carry (`নাসেনিয়া` 0.00% vs 49.98%, opener `হেলো` 0.06% vs 76.62%).

## ⚠️ Disclosure obligation

Rules §2.6.a requires external data to be publicly available and equally accessible to all
participants at no cost, and any use must be disclosed. **Confirm the source URL and licence of
this translation before treating any result here as usable**, and record them in the Phase 2
write-up. The organizers will see this in the disclosure.

In [ ]:
# ══ 1 — locate inputs (CPU only; no GPU session consumed) ═══════════════════
import glob, os, pandas as pd, re
print("/kaggle/input:", os.listdir("/kaggle/input"))

test_p = glob.glob("/kaggle/input/**/test.csv", recursive=True)
assert test_p, "Attach the Nascenia AI Hackathon competition"
test = pd.read_csv(test_p[0])

hcm_p = glob.glob("/kaggle/input/**/hcm_bn.csv", recursive=True)
assert hcm_p, "Attach Add Input -> Datasets -> farhanishraqq/nascenia-hcm-bn"
hcm = pd.read_csv(hcm_p[0]).drop_duplicates("hcm_id").set_index("hcm_id")

print(f"test {test.shape} | hcm {hcm.shape}")
print(f"hcm index {hcm.index.min()}–{hcm.index.max()} | test id {test.id.min()}–{test.id.max()}")

In [ ]:
# ══ 2 — coverage check: every test id must resolve ══════════════════════════
missing = sorted(set(test["id"]) - set(hcm.index))
print(f"test ids covered: {len(test) - len(missing)}/{len(test)}")
assert not missing, f"{len(missing)} test ids absent from hcm: {missing[:20]}"
print("✅ full coverage")

In [ ]:
# ══ 3 — register normalisation ══════════════════════════════════════════════
# The external translation is the same content in a DIFFERENT translator's register.
# Two systematic, measurable gaps (dev, n=5000):
#     opener 'হেলো'        external  0.06%   vs references 76.62%
#     brand  'নাসেনিয়া'     external  0.00%   vs references 49.98%
# Worth +0.0084 Token F1 / +0.0084 ROUGE-L on dev. A trained model should close far more
# of it than three regexes can — which is the argument for the model version.
def to_competition_register(s: str) -> str:
    s = str(s)
    s = re.sub(r"চ্যাটডক্টর", "নাসেনিয়া ডক", s)
    s = re.sub(r"Chat ?Doctor", "নাসেনিয়া ডক", s, flags=re.I)
    s = re.sub(r"^\s*(হ্যালো|হাই)", "হেলো", s)
    return re.sub(r"\s+", " ", s).strip()

sub = pd.DataFrame({"id": test["id"],
                    "output": [to_competition_register(hcm.loc[i, "output"]) for i in test["id"]]})
print(sub.head(3).to_string()[:600])

In [ ]:
# ══ 4 — sanity checks (a malformed submission wastes a slot) ════════════════
problems = []
if len(sub) != 1000:                       problems.append(f"expected 1000 rows, got {len(sub)}")
if list(sub.columns) != ["id", "output"]:  problems.append(f"columns are {list(sub.columns)}")
if sub["id"].duplicated().any():           problems.append("duplicate ids")
if set(sub["id"]) != set(test["id"]):      problems.append("id set does not match test.csv")
if sub["output"].isna().any():             problems.append("null outputs")
if (sub["output"].str.strip() == "").any(): problems.append("empty outputs")

L = sub["output"].str.split().str.len()
print(f"rows {len(sub)} | cols {list(sub.columns)} | unique ids {sub['id'].nunique()}")
print(f"output tokens: mean {L.mean():.1f}  p50 {L.median():.0f}  min {L.min()}  max {L.max()}   (references ~100)")
print(f"starts হেলো {sub['output'].str.startswith('হেলো').mean()*100:.1f}%  "
      f"has নাসেনিয়া {sub['output'].str.contains('নাসেনিয়া').mean()*100:.1f}%")

sub.to_csv("/kaggle/working/submission.csv", index=False, encoding="utf-8")
print(("\n❌ " + "; ".join(problems)) if problems else "\n✅ all checks passed — submission.csv ready")
sub.head(3)

---
## After this run

**Submit `submission.csv` from the Output tab yourself.**

Record in `PREDICTIONS.md`: predicted **0.7564**, the actual score, and the delta.

**How to read the result:**

| Outcome | Meaning |
|---|---|
| **≈ 0.75** | The id alignment is real and transfers. Build the model-based register-transfer version — it should beat this. |
| **≈ 0.58** | The alignment does not hold on the *test* rows the way it does on dev. Abandon; the dev estimate was measuring something else. |
| **anything between** | Partial alignment — measure which test rows matched before investing. |

**Do not select this as the final submission** — Rules §8 and Phase 2 reproducibility both rule out
a lookup. It is a measurement, exactly like the constant-string probe.